In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs with timestamp parsing (UTC)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_logs_df = pd.read_csv("error_logs.csv")
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# 1) metric.csv summary: group by (cmdb_id, kpi_name) -> count and global P95 of 'value'
metric_summary = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .sort_values('count', ascending=False)
    .head(20)
)

# Round p95 for compactness
metric_summary['p95_value'] = metric_summary['p95_value'].round(6)

# 2) trace.csv summary: group by (cmdb_id, trace_name)
trace_summary = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .sort_values('count', ascending=False)
    .head(20)
)
trace_summary['p95_value'] = trace_summary['p95_value'].round(6)

# 3) log.csv summary: group by (cmdb_id, log_name)
log_summary = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .sort_values('count', ascending=False)
    .head(20)
)
log_summary['p95_value'] = log_summary['p95_value'].round(6)

# 4) error_logs.csv: top 20 rows (most recent) and count of distinct cmdb_id values
error_top20 = error_logs_df.sort_values('timestamp', ascending=False)[['timestamp', 'cmdb_id', 'message']].head(20)
error_cmdb_count = error_logs_df['cmdb_id'].nunique()

# Display compact summaries (IPython will show these variables)
metric_summary, trace_summary, log_summary, error_top20, error_cmdb_count

```
Out[1]:
```
Summary of the telemetry aggregates (straightforward findings):

- General
  - All aggregated groups returned 25 samples each (top groups are tied at count = 25).
  - error_logs.csv is empty (no rows), and there are 0 distinct cmdb_id values in error_logs.

- metric.csv (top pairs by count — notable KPIs)
  - ts-admin-basic-info-service
    - mem: p95 ≈ 2.396e+08 (very large)
    - diskio: p95 ≈ 6.08e+04
    - cpu: p95 ≈ 0.445
    - socket: p95 ≈ 7.367
    - latency-50 / latency-90 also present
  - ts-admin-order-service and ts-admin-route-service and ts-admin-travel-service show similarly large mem and diskio p95 values and non-zero cpu/socket p95 values.
  - ts-voucher-service: socket p95 = 4.0

- trace.csv (top pairs by count — notable traces)
  - ts-travel2-service has many top trace series:
    - from_root.duration_p95 ≈ 0.6837 (highest trace p95 shown)
    - from_ts-admin-travel-service.duration_p95 ≈ 0.2037
    - various from/to service row_count values high (e.g., 2727.6, 340.0, 77.8)
  - ts-travel-service trace.to_ts-travel-service.duration_p95 ≈ 0.0582

- log.csv (top pairs by count — notable values)
  - Many services show log.row_count as a high p95 (examples):
    - ts-basic-service log.row_count p95 ≈ 2654.2
    - ts-travel-service log.row_count p95 ≈ 1976.4
    - ts-seat-service log.row_count p95 ≈ 1950.2
    - ts-auth-service log.row_count p95 ≈ 340.0
  - log.error_count p95 is 0.0 for the listed services (no aggregated log error counts).

Actionable interpretation (based on these aggregates)
- No error log entries were captured, so investigate metric/trace signals rather than log errors.
- Metric signals point to resource pressure on admin services (especially ts-admin-basic-info-service) — very high mem and high disk I/O p95 values — investigate memory/disk usage on those instances.
- Trace signals point to latency concentrated around ts-travel2-service (notably the from_root.duration_p95 ≈ 0.684s and from_admin-travel p95 ≈ 0.204s) and large row_count on some traces — investigate ts-travel2-service call paths and downstream load.
- Logs show high traffic (row_count) for basic/travel/seat services but no error counts — suggests load or throughput issues rather than application exceptions.

If you want, I can:
- produce ranked lists or filter these metrics/traces by a specific incident time window, or
- compute anomaly scores for the highlighted services/components.

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count     p95_value
372           ts-voucher-service      socket     25  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  4.452060e-01
1    ts-admin-basic-info-service      diskio     25  6.080373e+04
2    ts-admin-basic-info-service  latency-50     25  1.685200e-02
3    ts-admin-basic-info-service  latency-90     25  2.340900e-02
4    ts-admin-basic-info-service         mem     25  2.396362e+08
5    ts-admin-basic-info-service      socket     25  7.366667e+00
6    ts-admin-basic-info-service    workload     25  6.888430e-01
7         ts-admin-order-service         cpu     25  1.076700e-01
8         ts-admin-order-service      diskio     25  5.159549e+04
9         ts-admin-order-service         mem     25  1.954514e+08
10        ts-admin-order-service      socket     25  3.500000e+00
11        ts-admin-route-service         cpu     25  9.807700e-02
12        ts-admin-route-service      diskio     25  5.017227e+04
13        ts-admin-route-service         mem     25  1.956456e+08
14        ts-admin-route-service      socket     25  3.483333e+00
15       ts-admin-travel-service         cpu     25  3.743920e-01
16       ts-admin-travel-service      diskio     25  6.001908e+04
17       ts-admin-travel-service       error     25  0.000000e+00
18       ts-admin-travel-service  latency-50     25  1.930850e-01,                 cmdb_id                                        trace_name  count    p95_value
728  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25     0.013064
729  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25     0.025550
730  ts-travel2-service             trace.from_ts-seat-service.error_rate     25     0.000000
731  ts-travel2-service              trace.from_ts-seat-service.row_count     25   340.000000
732  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25     0.012844
733  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25     0.056139
734  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25     0.000000
735  ts-travel2-service           trace.from_ts-travel2-service.row_count     25  2727.600000
720  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25     0.098900
721  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25     0.203703
722  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25     0.000000
723  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25    10.800000
712   ts-travel-service          trace.to_ts-travel-service.duration_mean     25     0.018912
713   ts-travel-service           trace.to_ts-travel-service.duration_p95     25     0.058201
714   ts-travel-service             trace.to_ts-travel-service.error_rate     25     0.000000
715   ts-travel-service              trace.to_ts-travel-service.row_count     25  1765.200000
716  ts-travel2-service                     trace.from_root.duration_mean     25     0.376672
717  ts-travel2-service                      trace.from_root.duration_p95     25     0.683737
718  ts-travel2-service                        trace.from_root.error_rate     25     0.000000
719  ts-travel2-service                         trace.from_root.row_count     25    77.800000,                         cmdb_id         log_name  count  p95_value
0   ts-admin-basic-info-service  log.error_count     25        0.0
1   ts-admin-basic-info-service    log.row_count     25       32.8
2       ts-admin-travel-service  log.error_count     25        0.0
3       ts-admin-travel-service    log.row_count     25       32.4
10              ts-auth-service  log.error_count     25        0.0
11              ts-auth-service    log.row_count     25      340.0
13             ts-basic-service    log.row_count     25     2654.2
12             ts-basic-service  log.error_count     25        0.0
20           ts-consign-service  log.error_count     25        0.0
21           ts-consign-service    log.row_count     25       61.6
83            ts-travel-service    log.row_count     25     1976.4
82            ts-travel-service  log.error_count     25        0.0
60             ts-price-service  log.error_count     25        0.0
61             ts-price-service    log.row_count     25      295.2
65             ts-route-service    log.row_count     25     1064.6
64             ts-route-service  log.error_count     25        0.0
66              ts-seat-service  log.error_count     25        0.0
67              ts-seat-service    log.row_count     25     1950.2
46             ts-order-service  log.error_count     25        0.0
47             ts-order-service    log.row_count     25      413.8, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse or load metric.csv and parse timestamp (UTC)
try:
    metric_df  # use existing if available
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed and tz-aware
    if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64':
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    elif metric_df['timestamp'].dtype == 'O':
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter full series to candidate services
df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 for each (cmdb_id, kpi_name) using the full series (no time filtering)
global_p95 = (
    df_cand
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter to incident window (UTC)
start = pd.to_datetime("2024-01-23 05:22:46 UTC", utc=True)
end   = pd.to_datetime("2024-01-23 05:52:46 UTC", utc=True)
df_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()

# 3) Prepare window aggregations
# count_in_window and max_value_in_window
window_agg = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(count_in_window=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# Merge global_p95 into window rows to compute anomalies
df_window = df_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Flag anomalies where value >= global_p95 (global threshold from full series)
df_window['is_anomaly'] = df_window['value'] >= df_window['global_p95']

# Aggregate anomaly counts and earliest anomaly timestamp
anomaly_agg = (
    df_window[df_window['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(anomaly_count=('is_anomaly','sum'),
         earliest_anomaly_timestamp=('timestamp','min'))
)

# Format earliest_anomaly_timestamp as UTC ISO string with 'Z'
if not anomaly_agg.empty:
    anomaly_agg['earliest_anomaly_timestamp'] = anomaly_agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Combine global_p95, window_agg, and anomaly_agg
summary = global_p95.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NA for counts/max where there were no rows in window
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)
summary['max_value_in_window'] = summary['max_value_in_window']  # leave NaN if no rows
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp']  # NaN if none

# Round global_p95 for compactness
summary['global_p95'] = summary['global_p95'].round(6)

# 4) Filter to rows where anomaly_count > 0, sort and limit to top 20
result_summary = (
    summary[summary['anomaly_count'] > 0]
    .sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_p95','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]
    .head(20)
)

# Display the compact summary table
result_summary

```
Out[2]:
```
summary = (
    "Summary of findings (incident window 2024-01-23T05:22:46Z — 2024-01-23T05:52:46Z):\n\n"
    "1) 'error' metric (ts-auth-service, ts-order-service, ts-travel-service):\n"
    "- Each shows anomaly_count=25 with global_p95=0.0 and earliest occurrence at 2024-01-23T05:25:00Z.\n"
    "- This is likely an artifact: global P95=0 means every non-negative value meets >= global_p95, so treat these as spurious alerts rather than true error spikes.\n\n"
    "2) Memory (mem) anomalies (anomaly_count=2 each):\n"
    "- ts-order-service: global_p95 ≈ 2.940788e+08, earliest anomaly 2024-01-23T05:47:00Z, max in window ≈ 2.940957e+08.\n"
    "- ts-travel-service: global_p95 ≈ 2.718033e+08, earliest anomaly 2024-01-23T05:31:00Z, max ≈ 2.718204e+08.\n"
    "- ts-route-service: global_p95 ≈ 2.614513e+08, earliest anomaly 2024-01-23T05:42:00Z, max ≈ 2.659311e+08.\n"
    "- ts-train-service: global_p95 ≈ 2.534520e+08, earliest anomaly 2024-01-23T05:43:00Z, max ≈ 2.534742e+08.\n"
    "- ts-auth-service: global_p95 ≈ 2.453025e+08, earliest anomaly 2024-01-23T05:35:00Z, max ≈ 2.453262e+08.\n\n"
    "3) Disk I/O anomalies (anomaly_count=2 each):\n"
    "- ts-order-service: global_p95 ≈ 1.681099e+05, earliest anomaly 2024-01-23T05:25:00Z, max ≈ 1.239915e+06 (notable spike).\n"
    "- ts-travel-service: global_p95 ≈ 6.323700e+04, earliest anomaly 2024-01-23T05:32:00Z, max ≈ 6.460361e+04.\n"
    "- ts-train-service: global_p95 ≈ 5.931303e+04, earliest anomaly 2024-01-23T05:26:00Z, max ≈ 6.012191e+04.\n"
    "- ts-auth-service and ts-route-service also show small diskio anomalies (timestamps around 05:33–05:37Z).\n\n"
    "4) Workload / socket / cpu anomalies (anomaly_count=2 each):\n"
    "- workload: ts-route-service and ts-train-service showed brief increases (earliest ~05:30–05:41Z).\n"
    "- socket: ts-travel, ts-train, ts-order, ts-route exhibited socket spikes (earliest ~05:25–05:36Z).\n"
    "- cpu: ts-auth-service had a CPU anomaly (global_p95 ≈ 15.0833, earliest 05:26:00Z, max ≈ 15.20678).\n\n"
    "Interpretation & next steps:\n"
    "- Ignore the 'error' metric anomalies for now (global P95 == 0 causes every sample to appear anomalous).\n"
    "- Focus investigation on resource pressure: elevated memory and disk I/O in ts-order, ts-travel, ts-route, ts-train, and ts-auth during the window.\n    * Notable: ts-order had a large disk I/O spike (max ≈ 1.24e+06 at 05:25Z) and elevated memory at ~05:47Z.\n    * Memory anomalies are small-in-count but reach/briefly exceed historical P95 — investigate recent GC, memory allocation, or configuration changes.\n- Also check socket and workload spikes (may indicate increased traffic or connection churn) and correlate with traces/throughput.\n\n"
    "Recommended actions:\n"
    "1) Inspect detailed logs/traces around the listed timestamps for ts-order, ts-travel, ts-route, ts-train, ts-auth.\n"
    "2) Check host-level metrics (actual memory, disk I/O, network) and GC activity for impacted services.\n"
    "3) Dismiss the 'error' metric anomalies until the metric semantics/thresholds are corrected.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name    global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
2     ts-auth-service     error  0.000000e+00               25             25       2024-01-23T05:25:00Z         0.000000e+00
10   ts-order-service     error  0.000000e+00               25             25       2024-01-23T05:25:00Z         0.000000e+00
33  ts-travel-service     error  0.000000e+00               25             25       2024-01-23T05:25:00Z         0.000000e+00
13   ts-order-service       mem  2.940788e+08               25              2       2024-01-23T05:47:00Z         2.940957e+08
36  ts-travel-service       mem  2.718033e+08               25              2       2024-01-23T05:31:00Z         2.718204e+08
21   ts-route-service       mem  2.614513e+08               25              2       2024-01-23T05:42:00Z         2.659311e+08
28   ts-train-service       mem  2.534520e+08               25              2       2024-01-23T05:43:00Z         2.534742e+08
5     ts-auth-service       mem  2.453025e+08               25              2       2024-01-23T05:35:00Z         2.453262e+08
9    ts-order-service    diskio  1.681099e+05               25              2       2024-01-23T05:25:00Z         1.239915e+06
32  ts-travel-service    diskio  6.323700e+04               25              2       2024-01-23T05:32:00Z         6.460361e+04
25   ts-train-service    diskio  5.931303e+04               25              2       2024-01-23T05:26:00Z         6.012191e+04
1     ts-auth-service    diskio  5.783786e+04               25              2       2024-01-23T05:37:00Z         5.962354e+04
17   ts-route-service    diskio  5.736470e+04               25              2       2024-01-23T05:33:00Z         5.932842e+04
23   ts-route-service  workload  3.533208e+01               25              2       2024-01-23T05:30:00Z         3.597030e+01
30   ts-train-service  workload  2.205790e+01               25              2       2024-01-23T05:41:00Z         2.278855e+01
37  ts-travel-service    socket  1.809702e+01               25              2       2024-01-23T05:36:00Z         1.938333e+01
29   ts-train-service    socket  1.570539e+01               25              2       2024-01-23T05:32:00Z         1.620000e+01
14   ts-order-service    socket  1.548000e+01               25              2       2024-01-23T05:37:00Z         1.618333e+01
22   ts-route-service    socket  1.598667e+01               25              2       2024-01-23T05:25:00Z         1.600000e+01
0     ts-auth-service       cpu  1.508330e+01               25              2       2024-01-23T05:26:00Z         1.520678e+01```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing trace_df if available, else load and parse timestamp (UTC)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp is parsed and tz-aware
    if trace_df['timestamp'].dtype == 'int64' or trace_df['timestamp'].dtype == 'float64':
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    elif trace_df['timestamp'].dtype == 'O':
        # try to parse if not already datetime
        if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
            trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P95 for every (cmdb_id, trace_name) using full series
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter trace rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 05:22:46 UTC", utc=True)
end   = pd.to_datetime("2024-01-23 05:52:46 UTC", utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# Further restrict to candidate services
trace_window = trace_window[trace_window['cmdb_id'].isin(candidates)].copy()
global_p95_trace = global_p95_trace[global_p95_trace['cmdb_id'].isin(candidates)].copy()

# 3) For each (cmdb_id, trace_name) produce compact summary:
#    count_in_window, anomaly_count (value >= global_p95), earliest_anomaly_timestamp, max_value_in_window

# Window aggregations: count and max
window_agg = (
    trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(count_in_window=('value','size'),
         max_value_in_window=('value','max'))
)

# Merge global_p95 into the window rows to evaluate anomalies
trace_window = trace_window.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies where value >= global_p95 (global threshold computed on full series)
trace_window['is_anomaly'] = trace_window['value'] >= trace_window['global_p95']

# Aggregate anomaly counts and earliest anomaly timestamp
anomaly_agg = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anomaly','sum'),
         earliest_anomaly_timestamp=('timestamp','min'))
)

# Format earliest_anomaly_timestamp as UTC ISO string with 'Z'
if not anomaly_agg.empty:
    anomaly_agg['earliest_anomaly_timestamp'] = anomaly_agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Combine global_p95, window_agg, and anomaly_agg
summary = global_p95_trace.merge(window_agg, on=['cmdb_id','trace_name'], how='left')
summary = summary.merge(anomaly_agg, on=['cmdb_id','trace_name'], how='left')

# Fill NA for counts where there were no rows in window
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)
summary['max_value_in_window'] = summary['max_value_in_window']  # keep NaN if no rows
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp']  # NaN if none

# Round global_p95 for compactness
summary['global_p95'] = summary['global_p95'].round(6)

# 4) Filter to rows where anomaly_count > 0, sort and limit to top 20
trace_result = (
    summary[summary['anomaly_count'] > 0]
    .sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','trace_name','global_p95','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]
    .head(20)
)

trace_result

```
Out[3]:
```
Summary of findings (incident window 2024-01-23T05:22:46Z — 2024-01-23T05:52:46Z):

- All returned anomalies are trace error_rate series across the candidate services (ts-auth, ts-order, ts-route, ts-train, ts-travel).
- For every reported (cmdb_id, trace_name):
  - global P95 = 0.0
  - count_in_window = 25, anomaly_count = 25
  - earliest_anomaly_timestamp = 2024-01-23T05:25:00Z
  - max_value_in_window = 0.0

Interpretation and recommended action:
- These are spurious alerts: global P95 equals 0.0 and the trace values are 0.0, so the condition value >= global_p95 marks every sample as an "anomaly." This does not indicate real error traffic.
- Do not treat these trace error_rate hits as root cause. Instead:
  - Ignore or suppress error_rate alerts where global_p95 == 0 (or change anomaly rule to require value > global_p95 and/or global_p95 > 0).
  - Investigate non-zero signals found earlier (memory, disk I/O, socket, workload) for ts-order, ts-travel, ts-route, ts-train, ts-auth.
  - Check trace duration/error semantics and sampling to ensure error_rate metric is meaningful and thresholds are correct.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
2    ts-auth-service                       trace.from_root.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
6    ts-auth-service            trace.from_ts-auth-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
10   ts-auth-service              trace.to_ts-auth-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
14  ts-order-service                       trace.from_root.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
18  ts-order-service  trace.from_ts-inside-payment-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
22  ts-order-service           trace.from_ts-order-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
30  ts-order-service            trace.from_ts-seat-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
38  ts-order-service          trace.from_ts-travel-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
42  ts-order-service             trace.to_ts-order-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
46  ts-order-service           trace.to_ts-station-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
50  ts-route-service           trace.from_ts-basic-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
54  ts-route-service           trace.from_ts-route-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
58  ts-route-service          trace.from_ts-travel-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
62  ts-route-service         trace.from_ts-travel2-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
66  ts-route-service             trace.to_ts-route-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
70  ts-train-service           trace.from_ts-basic-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
74  ts-train-service           trace.from_ts-train-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
78  ts-train-service          trace.from_ts-travel-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
82  ts-train-service         trace.from_ts-travel2-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0
86  ts-train-service             trace.to_ts-train-service.error_rate         0.0               25             25       2024-01-23T05:25:00Z                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing log_df if available, else load and parse timestamp (UTC)
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed and tz-aware
    if not pd.api.types.is_datetime64_any_dtype(log_df['timestamp']):
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P95 for each (cmdb_id, log_name) using the full log series
global_p95_log = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter log rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 05:22:46 UTC", utc=True)
end   = pd.to_datetime("2024-01-23 05:52:46 UTC", utc=True)
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# Restrict to candidate services
log_window = log_window[log_window['cmdb_id'].isin(candidates)].copy()
global_p95_log = global_p95_log[global_p95_log['cmdb_id'].isin(candidates)].copy()

# 3) Window aggregations: count_in_window and max_value_in_window
window_agg = (
    log_window
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(count_in_window=('value','size'),
         max_value_in_window=('value','max'))
)

# Merge global_p95 into window rows to evaluate anomalies
log_window = log_window.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')

# Flag anomalies where value >= global_p95 (global threshold computed on full series)
log_window['is_anomaly'] = log_window['value'] >= log_window['global_p95']

# Aggregate anomaly counts and earliest anomaly timestamp
anomaly_agg = (
    log_window[log_window['is_anomaly']]
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(anomaly_count=('is_anomaly','sum'),
         earliest_anomaly_timestamp=('timestamp','min'))
)

# Format earliest_anomaly_timestamp as UTC ISO string with 'Z' if present
if not anomaly_agg.empty:
    anomaly_agg['earliest_anomaly_timestamp'] = anomaly_agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Combine global_p95, window_agg, and anomaly_agg
summary = global_p95_log.merge(window_agg, on=['cmdb_id','log_name'], how='left')
summary = summary.merge(anomaly_agg, on=['cmdb_id','log_name'], how='left')

# Fill NA for counts where there were no rows in window
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)
summary['max_value_in_window'] = summary['max_value_in_window']  # keep NaN if no rows
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp']  # NaN if none

# Round global_p95 for compactness
summary['global_p95'] = summary['global_p95'].round(6)

# 4) Filter to rows where anomaly_count > 0, sort and limit to top 20
log_result = (
    summary[summary['anomaly_count'] > 0]
    .sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','log_name','global_p95','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]
    .head(20)
)

log_result

```
Out[4]:
```
Summary (incident window 2024-01-23T05:22:46Z — 2024-01-23T05:52:46Z):

- log.error_count (ts-auth, ts-order, ts-route, ts-travel)
  - global P95 = 0.0, values = 0 in the window, anomaly_count = 25 for each.
  - These are spurious: global_p95 == 0 makes every non-negative sample appear anomalous. Do not treat these as real error spikes.

- log.row_count (meaningful anomalies)
  - ts-auth-service: global_p95 = 340.0, count_in_window = 25, anomaly_count = 3, earliest anomaly 2024-01-23T05:29:00Z, max row_count = 348.
  - ts-travel-service: global_p95 = 1976.4, count_in_window = 25, anomaly_count = 2, earliest 2024-01-23T05:34:00Z, max = 2021.
  - ts-route-service: global_p95 = 1064.6, count_in_window = 25, anomaly_count = 2, earliest 2024-01-23T05:34:00Z, max = 1133.
  - ts-order-service: global_p95 = 413.8, count_in_window = 25, anomaly_count = 2, earliest 2024-01-23T05:34:00Z, max = 422.

Interpretation and recommended next steps:
- Ignore the log.error_count hits (artifact of zero threshold).
- The meaningful signal is brief elevated log.row_count (throughput) for ts-auth (around 05:29) and for ts-travel/ts-route/ts-order (around 05:34). These indicate short bursts of increased request volume.
- Correlate these timestamps with metric and trace anomalies (memory, disk I/O, socket spikes previously identified) to determine if higher traffic caused resource pressure.
- Actions: check request rates, upstream traffic patterns, autoscaling/connection limits, and traces around the listed timestamps. Adjust alert logic to avoid global_p95==0 false positives.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0    ts-auth-service  log.error_count         0.0               25             25       2024-01-23T05:25:00Z                    0
2   ts-order-service  log.error_count         0.0               25             25       2024-01-23T05:25:00Z                    0
4   ts-route-service  log.error_count         0.0               25             25       2024-01-23T05:25:00Z                    0
6  ts-travel-service  log.error_count         0.0               25             25       2024-01-23T05:25:00Z                    0
1    ts-auth-service    log.row_count       340.0               25              3       2024-01-23T05:29:00Z                  348
7  ts-travel-service    log.row_count      1976.4               25              2       2024-01-23T05:34:00Z                 2021
5   ts-route-service    log.row_count      1064.6               25              2       2024-01-23T05:34:00Z                 1133
3   ts-order-service    log.row_count       413.8               25              2       2024-01-23T05:34:00Z                  422```
```

In [ ]:
# In[5]:

import pandas as pd

# Ensure metric_df is loaded and timestamp parsed (reuse if available)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to candidate services for global threshold calculation (rule 9)
df_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 for every (cmdb_id, kpi_name) using the full series
global_p95 = (
    df_cand
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 05:22:46 UTC", utc=True)
end   = pd.to_datetime("2024-01-23 05:52:46 UTC", utc=True)
df_window = df_cand[(df_cand['timestamp'] >= start) & (df_cand['timestamp'] <= end)].copy()

# Merge global_p95 into window rows to evaluate anomalies (using thresholds computed from full series)
df_window = df_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Flag anomalies where value >= global_p95
df_window['is_anomaly'] = df_window['value'] >= df_window['global_p95']

# 2b) For each (cmdb_id,kpi_name) compute count_in_window, anomaly_count, earliest_anomaly_timestamp, max_value_in_window
window_agg = (
    df_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(count_in_window=('value','size'),
         anomaly_count=('is_anomaly','sum'),
         earliest_anom_ts=('timestamp', lambda x: x[df_window.loc[x.index, 'is_anomaly']].min() if x.size>0 else pd.NaT),
         max_value_in_window=('value','max'))
)

# The lambda above may produce NaT for groups without anomalies; ensure formatting after merge

# Merge with global_p95 to ensure all pairs present (some pairs might have no rows in window)
summary = global_p95.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts and keep NaT for timestamps where appropriate
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
# max_value_in_window stays NaN if no rows
# earliest_anom_ts is timestamp or NaT; format if present
summary['earliest_anomaly_timestamp'] = pd.to_datetime(summary['earliest_anom_ts'])
summary = summary.drop(columns=['earliest_anom_ts'])

# Format earliest_anomaly_timestamp as ISO Z or keep NaN
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].where(~summary['earliest_anomaly_timestamp'].isna(), pd.NA)

# 3) Keep only rows where anomaly_count > 0 and global_p95 > 0
summary_filtered = summary[(summary['anomaly_count'] > 0) & (summary['global_p95'] > 0)].copy()

# Compute breach_ratio = max_value_in_window / global_p95
summary_filtered['breach_ratio'] = summary_filtered['max_value_in_window'] / summary_filtered['global_p95']

# 4) Prepare compact table with required columns, sort by breach_ratio desc, limit to top 20
result_df = (
    summary_filtered
    .loc[:, ['cmdb_id','kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','breach_ratio']]
    .sort_values('breach_ratio', ascending=False)
    .head(20)
)

# Round numeric columns for compactness
result_df['global_p95'] = result_df['global_p95'].round(6)
result_df['max_value_in_window'] = result_df['max_value_in_window'].round(6)
result_df['breach_ratio'] = result_df['breach_ratio'].round(6)

# Display the compact summary table
result_df

```
Out[5]:
```
Summary of metric breaches in the incident window (2024-01-23T05:22:46Z — 2024-01-23T05:52:46Z):

Key findings (ranked by breach ratio)
- ts-order-service — diskio
  - global P95 ≈ 168,109.9; max in window ≈ 1,239,915; breach_ratio ≈ 7.38
  - earliest anomaly: 2024-01-23T05:25:00Z
  - Priority: highest severity — very large disk I/O spike.

- ts-route-service — latency-90
  - global P95 ≈ 0.00912s; max ≈ 0.022512s; breach_ratio ≈ 2.47
  - earliest anomaly: 2024-01-23T05:26:00Z

- ts-route-service — cpu
  - global P95 ≈ 5.3366; max ≈ 11.07528; breach_ratio ≈ 2.08
  - earliest anomaly: 2024-01-23T05:25:00Z

- ts-travel-service — workload
  - global P95 ≈ 10.082; max ≈ 11.7037; breach_ratio ≈ 1.16
  - earliest anomaly: 2024-01-23T05:35:00Z

- ts-order-service — cpu, workload, latency-90, socket
  - moderate breaches (breach_ratio ~1.03–1.13), earliest anomalies around 05:25–05:41.

- ts-travel-service — latency-90, socket, diskio, cpu
  - several metrics exceed P95 modestly (breach_ratio ~1.02–1.07), earliest anomalies ~05:25–05:36.

- ts-train-service & ts-auth-service
  - workload/socket/cpu/diskio show modest breaches (breach_ratio ≈1.03) with earliest anomalies ~05:32–05:37.

Interpretation
- The standout issue is the ts-order-service disk I/O spike at ~05:25Z (over 7× above its historical P95). This is likely the most impactful resource event.
- Route service shows notable latency and CPU spikes around 05:25–05:26Z, suggesting latency/CPU pressure concurrent with the ts-order disk event.
- Other services (travel, train, auth) show smaller but consistent brief breaches across workload, socket, cpu, diskio — consistent with short bursts of higher traffic or downstream contention.
- Previous analyses showed spurious zero-threshold alerts for error_rate and log.error_count; ignore those for root-cause.

Recommended next steps (practical troubleshooting)
1. Prioritize ts-order-service:
   - Investigate what caused the large disk I/O at ~05:25 (heavy queries, backups, compaction, disk contention, or storage subsystem issues).
   - Check DB/storage metrics, slow queries, I/O wait, and host-level disk I/O counters at that time.

2. Correlate ts-route latency/cpu spikes (~05:25–05:26) with the ts-order disk event:
   - Determine if route requests depended on order service or shared infrastructure (DB, cache, disk).

3. For other services (ts-travel, ts-train, ts-auth):
   - Inspect traces and logs around their earliest_anomaly timestamps (05:25–05:37) for increased request rates, timeouts, thread-pool saturation, or connection churn.
   - Verify if socket/workload spikes were upstream traffic bursts (log.row_count anomalies earlier) causing resource pressure.

4. Mitigations:
   - If confirmed storage I/O bottleneck, throttle heavy jobs, optimize queries, add IOPS capacity, or move high-load operations off peak path.
   - Temporarily scale services or increase resources for affected hosts while investigating.
   - Fix alert thresholds to avoid P95==0 false positives for error metrics.

If you want, I can:
- produce the full list of affected (cmdb_id, kpi_name) with their exact numeric values, or
- pull traces/logs around the top timestamps (05:25–05:37) for targeted correlation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name     global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  breach_ratio
9    ts-order-service      diskio  168109.915853              2       2024-01-23T05:25:00Z         1.239915e+06      7.375619
20   ts-route-service  latency-90       0.009120              2       2024-01-23T05:26:00Z         2.251200e-02      2.468417
16   ts-route-service         cpu       5.336563              2       2024-01-23T05:25:00Z         1.107528e+01      2.075359
38  ts-travel-service    workload      10.081993              2       2024-01-23T05:35:00Z         1.170370e+01      1.160852
8    ts-order-service         cpu       4.530798              2       2024-01-23T05:25:00Z         5.132342e+00      1.132768
15   ts-order-service    workload       6.949503              2       2024-01-23T05:41:00Z         7.751936e+00      1.115466
12   ts-order-service  latency-90       0.446440              2       2024-01-23T05:29:00Z         4.899940e-01      1.097559
35  ts-travel-service  latency-90       0.283521              2       2024-01-23T05:25:00Z         3.039200e-01      1.071948
37  ts-travel-service      socket      18.097021              2       2024-01-23T05:36:00Z         1.938333e+01      1.071079
19   ts-route-service  latency-50       0.004867              2       2024-01-23T05:26:00Z         5.111000e-03      1.050090
14   ts-order-service      socket      15.480000              2       2024-01-23T05:37:00Z         1.618333e+01      1.045435
31  ts-travel-service         cpu       3.598732              2       2024-01-23T05:36:00Z         3.748721e+00      1.041678
17   ts-route-service      diskio   57364.697822              2       2024-01-23T05:33:00Z         5.932842e+04      1.034232
30   ts-train-service    workload      22.057900              2       2024-01-23T05:41:00Z         2.278855e+01      1.033124
29   ts-train-service      socket      15.705390              2       2024-01-23T05:32:00Z         1.620000e+01      1.031493
1     ts-auth-service      diskio   57837.855286              2       2024-01-23T05:37:00Z         5.962354e+04      1.030874
11   ts-order-service  latency-50       0.013304              2       2024-01-23T05:29:00Z         1.366300e-02      1.026933
32  ts-travel-service      diskio   63237.001758              2       2024-01-23T05:32:00Z         6.460361e+04      1.021611
4     ts-auth-service  latency-90       0.807849              2       2024-01-23T05:30:00Z         8.234670e-01      1.019333
23   ts-route-service    workload      35.332077              2       2024-01-23T05:30:00Z         3.597030e+01      1.018064```
```